# Pandas for Data Engineering

While navigating through this notebook keep a Data Engineering mindset: correctness, consistency, and reproducibility.


## 1) Setup: imports, logging, and project folders

In Data Engineering work, having a predictable folder structure makes your pipeline easier to debug and rerun safely.

We will use:
- `data/raw/`   : files as received (no guarantees)
- `data/bronze/`: minimally parsed + quarantine outputs
- `data/silver/`: cleaned and validated outputs
- `data/output/`: consumer-facing reports

Dependencies:
- `pandas`, `numpy`
- Parquet support requires one engine:
  - Recommended: `pip install pyarrow`
  - Alternative: `pip install fastparquet`


In [1]:

import json
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("pandas-de")

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
OUTPUT_DIR = DATA_DIR / "output"

for d in [RAW_DIR, BRONZE_DIR, SILVER_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

(RAW_DIR, BRONZE_DIR, SILVER_DIR, OUTPUT_DIR)


(PosixPath('/workspaces/PythonAcademy/Numpy_Pandas/data/raw'),
 PosixPath('/workspaces/PythonAcademy/Numpy_Pandas/data/bronze'),
 PosixPath('/workspaces/PythonAcademy/Numpy_Pandas/data/silver'),
 PosixPath('/workspaces/PythonAcademy/Numpy_Pandas/data/output'))

## 2) Pandas basics in a pipeline context: Series and DataFrame

Pandas provides two primary data structures:
- **Series**: one-dimensional labeled array
- **DataFrame**: two-dimensional labeled table (similar to a SQL table)

In Data Engineering, a DataFrame usually represents a batch of records to clean, validate, and publish.

Key inspection tools you should default to:
- `head()` to sample rows
- `shape` for row/column count
- `info()` for dtypes and null counts
- `describe(include="all")` for quick stats


In [2]:
# Series example
s1 = pd.Series([3, 5, 1, 2, 7])
s1


0    3
1    5
2    1
3    2
4    7
dtype: int64

In [3]:
# DataFrame example
df_example = pd.DataFrame({"x": [1, 2, 3], "y": ["a", "b", "c"]})
df_example


,x,y
0,1,a
1,2,b
2,3,c


In [4]:
df_example.head(2)

,x,y
0,1,a
1,2,b


In [5]:
df_example.shape

(3, 2)

In [6]:
df_example.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   x       3 non-null      int64
 1   y       3 non-null      str  
dtypes: int64(1), str(1)
memory usage: 180.0 bytes


In [7]:
df_example.describe(include="all")

,x,y
count,3.0,3
unique,NaN,3
top,NaN,a
freq,NaN,1
mean,2.0,NaN
std,1.0,NaN
min,1.0,NaN
25%,1.5,NaN
50%,2.0,NaN
75%,2.5,NaN


## 3) Build a consistent dataset theme: people master + events log

We will create two small datasets and then write them to disk.

People master (`people_df`):
- stable key: `person_id`
- descriptive attributes: `name`, `city`, `gender`
- semi-business fields: `salary_usd`, `signup_date`

Events log (`events_df`):
- stable key: `event_id`
- references people via `person_id`
- schema drift: extra fields (e.g., `campaign_id`, `reason`)
- missing field case: one event missing `event_type`
- unknown key case: one event references an unknown `person_id` (to demonstrate joins)


In [8]:

people_records = [
    {"person_id": 1001, "name": "Alice",   "age": 29,  "gender": "F", "city": "Oxfordshire",   "email": "alice@example.com",   "signup_date": "2024-01-15"},
    {"person_id": 1002, "name": "Bob",     "age": 83,  "gender": "M", "city": "Marshall",      "email": None,                  "signup_date": "2023-11-03"},
    {"person_id": 1003, "name": "Charlie", "age": 34,  "gender": "M", "city": "Kansas City",   "email": "charlie@example.com", "signup_date": "2024-02-10"},
    {"person_id": 1004, "name": "Bastian", "age": 12,  "gender": "M", "city": "De Forest",     "email": "bastian@example.com", "signup_date": "2024-07-01"},
    {"person_id": 1005, "name": "Ella",    "age": 79,  "gender": "F", "city": "Newport News",  "email": "ella@example.com",    "signup_date": "2022-06-20"},
    {"person_id": 1006, "name": "Jaco",    "age": 35,  "gender": "M", "city": "Norristown",    "email": "jaco@example.com",    "signup_date": "2024-04-05"},
    {"person_id": 1007, "name": "Ash",     "age": None,"gender": "M", "city": "Pallet Town",   "email": "ash@example.com",     "signup_date": "2024-05-18"},
    {"person_id": 1010, "name": "Alice",   "age": 30,  "gender": "F", "city": "Oxfordshire",   "email": "alice2@example.com",  "signup_date": "2025-01-10"},
]

people_df = pd.DataFrame(people_records)
people_df["salary_usd"] = [85000, np.nan, 95000, None, 72000, 110000, 65000, 90000]

events_records = [
    {"event_id": "e-0001", "person_id": 1001, "event_type": "login",    "event_ts": "2025-02-01T08:10:00", "source": "web"},
    {"event_id": "e-0002", "person_id": 1001, "event_type": "purchase", "event_ts": "2025-02-01T08:22:00", "source": "web", "amount_usd": 19.99},
    {"event_id": "e-0003", "person_id": 1002, "event_type": "login",    "event_ts": "2025-02-03T12:00:00", "source": "mobile"},
    {"event_id": "e-0004", "person_id": 9999, "event_type": "login",    "event_ts": "2025-02-03T12:05:00", "source": "web"},  # unknown person_id
    {"event_id": "e-0005", "person_id": 1003, "event_type": "purchase", "event_ts": "2025-02-05T17:45:00", "source": "mobile", "amount_usd": 120.00},
    {"event_id": "e-0006", "person_id": 1005, "event_type": "refund",   "event_ts": "2025-02-06T09:20:00", "source": "web", "amount_usd": 120.00, "reason": "duplicate_charge"},
    {"event_id": "e-0007", "person_id": 1006, "event_type": "login",    "event_ts": "2025-02-06T10:00:00", "source": "web", "campaign_id": "cmp-55"},
    {"event_id": "e-0008", "person_id": 1007,                 "event_ts": "2025-02-07T11:30:00", "source": "mobile"},  # missing event_type
]

events_df = pd.DataFrame(events_records)

people_df.head(), events_df.head()


(   person_id     name   age gender          city                email  \
 0       1001    Alice  29.0      F   Oxfordshire    alice@example.com   
 1       1002      Bob  83.0      M      Marshall                  NaN   
 2       1003  Charlie  34.0      M   Kansas City  charlie@example.com   
 3       1004  Bastian  12.0      M     De Forest  bastian@example.com   
 4       1005     Ella  79.0      F  Newport News     ella@example.com   
 
   signup_date  salary_usd  
 0  2024-01-15     85000.0  
 1  2023-11-03         NaN  
 2  2024-02-10     95000.0  
 3  2024-07-01         NaN  
 4  2022-06-20     72000.0  ,
   event_id  person_id event_type             event_ts  source  amount_usd  \
 0   e-0001       1001      login  2025-02-01T08:10:00     web         NaN   
 1   e-0002       1001   purchase  2025-02-01T08:22:00     web       19.99   
 2   e-0003       1002      login  2025-02-03T12:00:00  mobile         NaN   
 3   e-0004       9999      login  2025-02-03T12:05:00     web     

## 4) File I/O

A common beginner pitfall is referencing files that don't exist. Here we create them.

We will write:
- `data/raw/people.csv`
- `data/raw/events.jsonl`

We also add one malformed JSON line at the end of the JSONL file to simulate a real ingestion failure.


In [9]:

people_csv_path = RAW_DIR / "people.csv"
events_jsonl_path = RAW_DIR / "events.jsonl"

people_df.to_csv(people_csv_path, index=False)

with events_jsonl_path.open("w", encoding="utf-8") as f:
    for rec in events_records:
        f.write(json.dumps(rec) + "\n")
    # Malformed JSON line (missing closing brace)
    f.write('{"event_id":"e-9999","person_id":1001,"event_type":"login","event_ts":"2025-02-08T08:00:00"\n')

people_csv_path, events_jsonl_path


(PosixPath('/workspaces/PythonAcademy/Numpy_Pandas/data/raw/people.csv'),
 PosixPath('/workspaces/PythonAcademy/Numpy_Pandas/data/raw/events.jsonl'))

### Exercise 4.1 (TODO): Read people.csv and inspect schema

Task:
- Read `people_csv_path` into `raw_people_df`
- Create `people_columns` as a list of column names
- Create `people_dtypes` as a dict mapping column -> dtype string


In [10]:
# TODO
raw_people_df = pd.read_csv(people_csv_path)
people_columns = raw_people_df.columns.to_list()
people_dtypes = {col: str(dt) for col,dt in raw_people_df.dtypes.items()}

display(raw_people_df.head(2))
display(people_columns)
display(people_dtypes)


,person_id,name,age,gender,city,email,signup_date,salary_usd
0,1001,Alice,29.0,F,Oxfordshire,alice@example.com,2024-01-15,85000.0
1,1002,Bob,83.0,M,Marshall,NaN,2023-11-03,NaN


['person_id',
 'name',
 'age',
 'gender',
 'city',
 'email',
 'signup_date',
 'salary_usd']

{'person_id': 'int64',
 'name': 'str',
 'age': 'float64',
 'gender': 'str',
 'city': 'str',
 'email': 'str',
 'signup_date': 'str',
 'salary_usd': 'float64'}

In [11]:
dir(raw_people_df)

['T',
 '_AXIS_LEN',
 '_AXIS_ORDERS',
 '_AXIS_TO_AXIS_NUMBER',
 '_HANDLED_TYPES',
 '__abs__',
 '__add__',
 '__and__',
 '__annotations__',
 '__array__',
 '__array_priority__',
 '__array_ufunc__',
 '__arrow_c_stream__',
 '__bool__',
 '__class__',
 '__contains__',
 '__copy__',
 '__dataframe__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__divmod__',
 '__doc__',
 '__eq__',
 '__finalize__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__ifloordiv__',
 '__imod__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__invert__',
 '__ior__',
 '__ipow__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__len__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__or__',
 '__pandas_priority__',
 '__pos__',
 '__pow__',
 '__radd__',
 '__rand__',
 '__rdivmod__',
 '__reduce__',
 '

## 5) Indexing and selection: boolean masks, `.loc`, `.iloc`

Most pipeline selection work should use:
- a boolean mask
- `.loc[mask, columns]`

This keeps code explicit and avoids chained indexing bugs.


In [12]:
people_work_df = raw_people_df.copy() # Shallow copy and Deep Copy
people_work_df.head()


,person_id,name,age,gender,city,email,signup_date,salary_usd
0,1001,Alice,29.0,F,Oxfordshire,alice@example.com,2024-01-15,85000.0
1,1002,Bob,83.0,M,Marshall,NaN,2023-11-03,NaN
2,1003,Charlie,34.0,M,Kansas City,charlie@example.com,2024-02-10,95000.0
3,1004,Bastian,12.0,M,De Forest,bastian@example.com,2024-07-01,NaN
4,1005,Ella,79.0,F,Newport News,ella@example.com,2022-06-20,72000.0


In [13]:
adult_mask = pd.to_numeric(people_work_df["age"], errors="coerce").fillna(-1) >= 18
adults_df = people_work_df.loc[adult_mask, ["person_id","name","age","city"]]
adults_df


,person_id,name,age,city
0,1001,Alice,29.0,Oxfordshire
1,1002,Bob,83.0,Marshall
2,1003,Charlie,34.0,Kansas City
4,1005,Ella,79.0,Newport News
5,1006,Jaco,35.0,Norristown
7,1010,Alice,30.0,Oxfordshire


### Exercise 5.1 (TODO): Filter high-salary people with `.loc`

Task:
- Create `high_salary_df` where `salary_usd >= 90000`
- Keep columns: `person_id`, `name`, `salary_usd`
- Use a boolean mask + `.loc`


In [14]:
high_salary_mask = pd.to_numeric(people_df["salary_usd"], errors='coerce').fillna(-1) >= 90000
high_salary_df = people_df.loc[high_salary_mask,["person_id","name","salary_usd"]]
high_salary_df


,person_id,name,salary_usd
2,1003,Charlie,95000.0
5,1006,Jaco,110000.0
7,1010,Alice,90000.0


### Exercise 5.1 (Checks)


In [15]:
assert list(high_salary_df.columns) == ["person_id","name","salary_usd"]
assert (pd.to_numeric(high_salary_df["salary_usd"], errors="coerce") >= 90000).all()


## 6) Cleaning: types, missing values, duplicates

Cleaning is about making data consistent and safe to use.

We will:
1) normalize types with coercion (`to_numeric`, `to_datetime`)
2) handle missing values intentionally
3) deduplicate by key (not by whole row)


In [16]:
filtered_df = people_work_df[people_work_df["age"].fillna(-1) > 30]
dropped_df = people_work_df.drop(columns=["city"])
renamed_df = people_work_df.rename(columns={"age": "years"})
(filtered_df.head(), dropped_df.head(), renamed_df.head())


(   person_id     name   age gender          city                email  \
 1       1002      Bob  83.0      M      Marshall                  NaN   
 2       1003  Charlie  34.0      M   Kansas City  charlie@example.com   
 4       1005     Ella  79.0      F  Newport News     ella@example.com   
 5       1006     Jaco  35.0      M    Norristown     jaco@example.com   
 
   signup_date  salary_usd  
 1  2023-11-03         NaN  
 2  2024-02-10     95000.0  
 4  2022-06-20     72000.0  
 5  2024-04-05    110000.0  ,
    person_id     name   age gender                email signup_date  \
 0       1001    Alice  29.0      F    alice@example.com  2024-01-15   
 1       1002      Bob  83.0      M                  NaN  2023-11-03   
 2       1003  Charlie  34.0      M  charlie@example.com  2024-02-10   
 3       1004  Bastian  12.0      M  bastian@example.com  2024-07-01   
 4       1005     Ella  79.0      F     ella@example.com  2022-06-20   
 
    salary_usd  
 0     85000.0  
 1         NaN

In [17]:
clean_people_df = people_work_df.copy()
clean_people_df["signup_date"] = pd.to_datetime(clean_people_df["signup_date"], errors="coerce")
clean_people_df["age"] = pd.to_numeric(clean_people_df["age"], errors="coerce")
clean_people_df["salary_usd"] = pd.to_numeric(clean_people_df["salary_usd"], errors="coerce")
clean_people_df["email"] = clean_people_df["email"].fillna("unknown@example.com")
clean_people_df["salary_usd"] = clean_people_df["salary_usd"].fillna(0.0)
clean_people_df.isna().mean().sort_values(ascending=False)


age            0.125
person_id      0.000
name           0.000
gender         0.000
city           0.000
email          0.000
signup_date    0.000
salary_usd     0.000
dtype: float64

### Exercise 6.1 (TODO): Fill missing age using the median


In [18]:
pd.to_numeric(people_df["age"],errors="coerce").fillna(people_df["age"].median())

0    29.0
1    83.0
2    34.0
3    12.0
4    79.0
5    35.0
6    34.0
7    30.0
Name: age, dtype: float64

In [19]:
# TODO
age_filled_people_df = clean_people_df.copy()
age_filled_people_df["age"] = pd.to_numeric(clean_people_df["age"],errors="coerce").fillna(people_df["age"].median())


### Exercise 6.1 (Checks)


In [20]:
assert pd.api.types.is_numeric_dtype(age_filled_people_df["age"])
assert age_filled_people_df["age"].isna().sum() == 0


In [35]:
dup_row = age_filled_people_df.loc[age_filled_people_df["person_id"] == 1003].copy()
people_with_dup = pd.concat([age_filled_people_df, dup_row], ignore_index=True)
people_with_dup[people_with_dup["person_id"] == 1003]


,person_id,name,age,gender,city,email,signup_date,salary_usd
2,1003,Charlie,34.0,M,Kansas City,charlie@example.com,2024-02-10,95000.0
8,1003,Charlie,34.0,M,Kansas City,charlie@example.com,2024-02-10,95000.0


### Exercise 6.2 (TODO): Deduplicate by `person_id`


In [36]:
# TODO
people_deduped_df = people_with_dup
people_deduped_df.drop_duplicates(["person_id"],inplace=True, keep="last")
people_deduped_df

,person_id,name,age,gender,city,email,signup_date,salary_usd
0,1001,Alice,29.0,F,Oxfordshire,alice@example.com,2024-01-15,85000.0
1,1002,Bob,83.0,M,Marshall,unknown@example.com,2023-11-03,0.0
3,1004,Bastian,12.0,M,De Forest,bastian@example.com,2024-07-01,0.0
4,1005,Ella,79.0,F,Newport News,ella@example.com,2022-06-20,72000.0
5,1006,Jaco,35.0,M,Norristown,jaco@example.com,2024-04-05,110000.0
6,1007,Ash,34.0,M,Pallet Town,ash@example.com,2024-05-18,65000.0
7,1010,Alice,30.0,F,Oxfordshire,alice2@example.com,2025-01-10,90000.0
8,1003,Charlie,34.0,M,Kansas City,charlie@example.com,2024-02-10,95000.0


### Exercise 6.2 (Checks)


In [25]:
assert people_deduped_df.duplicated(subset=["person_id"]).sum() == 0
assert people_deduped_df["person_id"].nunique() == people_deduped_df.shape[0]


## 7) Data contracts with `dataclass` (production-like validation)

We validate `people` with explicit checks that raise `ValueError` with clear messages.


In [37]:

@dataclass(frozen=True)
class PeopleContract:
    required_columns: Tuple[str, ...] = ("person_id","name","age","gender","city","email","signup_date","salary_usd")
    key_column: str = "person_id"
    gender_allowed: Tuple[str, ...] = ("F","M")
    age_min: float = 0.0
    age_max: float = 120.0
    salary_min: float = 0.0

    def validate(self, df: pd.DataFrame) -> None:
        missing = set(self.required_columns) - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")

        if df[self.key_column].isna().any():
            raise ValueError(f"{self.key_column} must not be null")

        if not df[self.key_column].is_unique:
            raise ValueError(f"{self.key_column} must be unique")

        if not pd.api.types.is_numeric_dtype(df["age"]):
            raise ValueError("age must be numeric")

        if not df["age"].between(self.age_min, self.age_max).all():
            raise ValueError("age contains values outside allowed range")

        if not set(df["gender"].dropna().unique()).issubset(set(self.gender_allowed)):
            raise ValueError("gender contains unexpected values")

        if not pd.api.types.is_datetime64_any_dtype(df["signup_date"]):
            raise ValueError("signup_date must be datetime")

        if not pd.api.types.is_numeric_dtype(df["salary_usd"]):
            raise ValueError("salary_usd must be numeric")

        if (df["salary_usd"] < self.salary_min).any():
            raise ValueError("salary_usd must be non-negative")

people_contract = PeopleContract()


In [38]:
people_validated_df = people_deduped_df.copy()
people_validated_df["signup_date"] = pd.to_datetime(people_validated_df["signup_date"], errors="coerce")
people_validated_df["age"] = pd.to_numeric(people_validated_df["age"], errors="coerce")
people_validated_df["salary_usd"] = pd.to_numeric(people_validated_df["salary_usd"], errors="coerce").fillna(0.0)

people_contract.validate(people_validated_df)
logger.info("PeopleContract validation: OK")

people_validated_df.head()


INFO | PeopleContract validation: OK


,person_id,name,age,gender,city,email,signup_date,salary_usd
0,1001,Alice,29.0,F,Oxfordshire,alice@example.com,2024-01-15,85000.0
1,1002,Bob,83.0,M,Marshall,unknown@example.com,2023-11-03,0.0
3,1004,Bastian,12.0,M,De Forest,bastian@example.com,2024-07-01,0.0
4,1005,Ella,79.0,F,Newport News,ella@example.com,2022-06-20,72000.0
5,1006,Jaco,35.0,M,Norristown,jaco@example.com,2024-04-05,110000.0


## 8) Robust ingestion of JSON Lines with quarantine


In [39]:
def read_jsonl_with_quarantine(path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    valid: List[Dict[str, Any]] = []
    bad: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            raw = line.rstrip("\n")
            if not raw.strip():
                continue
            try:
                valid.append(json.loads(raw))
            except Exception as e:
                bad.append({"line_no": line_no, "error": str(e), "raw_line": raw})
    return pd.DataFrame(valid), pd.DataFrame(bad)

events_ok_df, events_bad_df = read_jsonl_with_quarantine(events_jsonl_path)
logger.info(f"Events parsed: ok={events_ok_df.shape[0]} bad={events_bad_df.shape[0]}")

quarantine_path = BRONZE_DIR / "events_quarantine.csv"
events_bad_df.to_csv(quarantine_path, index=False)

events_ok_df.head(), events_bad_df


INFO | Events parsed: ok=8 bad=1


(  event_id  person_id event_type             event_ts  source  amount_usd  \
 0   e-0001       1001      login  2025-02-01T08:10:00     web         NaN   
 1   e-0002       1001   purchase  2025-02-01T08:22:00     web       19.99   
 2   e-0003       1002      login  2025-02-03T12:00:00  mobile         NaN   
 3   e-0004       9999      login  2025-02-03T12:05:00     web         NaN   
 4   e-0005       1003   purchase  2025-02-05T17:45:00  mobile      120.00   
 
   reason campaign_id  
 0    NaN         NaN  
 1    NaN         NaN  
 2    NaN         NaN  
 3    NaN         NaN  
 4    NaN         NaN  ,
    line_no                                              error  \
 0        9  Expecting ',' delimiter: line 1 column 92 (cha...   
 
                                             raw_line  
 0  {"event_id":"e-9999","person_id":1001,"event_t...  )

### Exercise 8.1 (TODO): Standardize events


In [42]:
# TODO
# Create events_validated_df from raw_events_df:
#   - Parse events_ts as datetime (Coerce errors with NaT)
#   - fill missing event_type with "unknown"

events_validated_df = events_ok_df
events_validated_df["event_ts"] = pd.to_datetime(events_ok_df["event_ts"], errors="coerce")
events_validated_df["event_type"] = events_validated_df["event_type"].fillna("unknown")


### Exercise 8.1 (Checks)


In [43]:

assert isinstance(events_validated_df, pd.DataFrame)
assert {"event_id","person_id","event_ts","event_type"}.issubset(set(events_validated_df.columns))
assert events_validated_df["event_id"].notna().all()
assert events_validated_df["person_id"].notna().all()
assert events_validated_df["event_ts"].notna().all()
assert events_validated_df["event_type"].isna().sum() == 0
assert pd.api.types.is_datetime64_any_dtype(events_validated_df["event_ts"])


## 9) Writing curated outputs (silver) and idempotency


In [45]:
!pip install pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 46.1 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [fastparquet] [fastparquet]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [50]:
import pyarrow as pa
people_silver_path = SILVER_DIR / "people.parquet"
events_silver_path = SILVER_DIR / "events.parquet"

people_validated_df.to_parquet(people_silver_path, index=False, engine='fastparquet')
events_validated_df.to_parquet(events_silver_path, index=False, engine='fastparquet')

people_silver_df = pd.read_parquet(people_silver_path, engine="fastparquet")
events_silver_df = pd.read_parquet(events_silver_path, engine="fastparquet")

people_silver_df.shape, events_silver_df.shape


((8, 8), (8, 8))

### Exercise 9.1 (TODO): Round-trip row counts


In [51]:
# TODO
people_row_count = people_silver_df.shape[0]
events_row_count = events_silver_df.shape[0]


### Exercise 9.1 (Checks)


In [52]:
assert isinstance(people_row_count, int) and people_row_count > 0
assert isinstance(events_row_count, int) and events_row_count > 0
assert people_row_count == people_silver_df.shape[0]
assert events_row_count == events_silver_df.shape[0]


## 10) Joining and aggregations: enrich events and compute metrics


In [53]:
enriched_events_df = events_silver_df.merge(
    people_silver_df[["person_id","city","gender"]],
    on="person_id",
    how="left",
    validate="many_to_one",
)
assert enriched_events_df.shape[0] == events_silver_df.shape[0]
unmatched = enriched_events_df["city"].isna().sum()
logger.info(f"Unmatched events (unknown person_id): {unmatched}")
enriched_events_df.head()


INFO | Unmatched events (unknown person_id): 1


,event_id,person_id,event_type,event_ts,source,amount_usd,reason,campaign_id,city,gender
0,e-0001,1001,login,2025-02-01 08:10:00,web,NaN,None,None,Oxfordshire,F
1,e-0002,1001,purchase,2025-02-01 08:22:00,web,19.99,None,None,Oxfordshire,F
2,e-0003,1002,login,2025-02-03 12:00:00,mobile,NaN,None,None,Marshall,M
3,e-0004,9999,login,2025-02-03 12:05:00,web,NaN,None,None,NaN,NaN
4,e-0005,1003,purchase,2025-02-05 17:45:00,mobile,120.00,None,None,Kansas City,M


In [54]:
tmp = enriched_events_df.copy()
tmp["city"] = tmp["city"].fillna("UNKNOWN")

event_counts = (
    tmp.groupby(["city","event_type"], as_index=False)
       .agg(event_count=("event_id","count"))
       .sort_values(["city","event_type"])
)
event_counts


,city,event_type,event_count
0,Kansas City,purchase,1
1,Marshall,login,1
2,Newport News,refund,1
3,Norristown,login,1
4,Oxfordshire,login,1
5,Oxfordshire,purchase,1
6,Pallet Town,unknown,1
7,UNKNOWN,login,1


### Exercise 10.1 (TODO): Aggregation totals


In [65]:
# TODO
total_events_from_counts = int(event_counts["event_count"].astype(int).sum())
print(total_events_from_counts)
total_events_raw = events_silver_df.shape[0]
print(total_events_raw)


8
8


### Exercise 10.1 (Checks)


In [66]:
assert isinstance(total_events_from_counts, int)
assert isinstance(total_events_raw, int)
assert total_events_from_counts == total_events_raw


In [ ]:

pivot_report = event_counts.pivot_table(
    index="city",
    columns="event_type",
    values="event_count",
    aggfunc="sum",
    fill_value=0,
).reset_index()
pivot_report


### Exercise 10.2 (TODO): Melt pivot report


In [ ]:
# TODO
tidy_again_df = None


### Exercise 10.2 (Checks)


In [ ]:
assert set(tidy_again_df.columns) == {"city","event_type","event_count"}
assert tidy_again_df.shape[0] == pivot_report.shape[0] * (pivot_report.shape[1] - 1)


## 11) Quick EDA: descriptive statistics


In [ ]:
people_silver_df.describe(include="all").T


In [ ]:
null_rates = people_silver_df.isna().mean().sort_values(ascending=False)
null_rates


## 12) Large datasets: chunking pattern (runnable)


In [ ]:
large_csv_path = RAW_DIR / "people_large.csv"
large_df = pd.concat([people_silver_df] * 3000, ignore_index=True)
large_df.to_csv(large_csv_path, index=False)
large_csv_path, large_df.shape


In [ ]:
def process_chunk(chunk: pd.DataFrame) -> Dict[str, int]:
    placeholder_emails = int((chunk["email"] == "unknown@example.com").sum())
    missing_age = int(chunk["age"].isna().sum())
    return {"rows": int(len(chunk)), "placeholder_emails": placeholder_emails, "missing_age": missing_age}

chunk_stats: List[Dict[str, int]] = []
for chunk in pd.read_csv(large_csv_path, chunksize=10_000):
    chunk_stats.append(process_chunk(chunk))

chunk_stats[:3], len(chunk_stats)


### Exercise 12.1 (TODO): Rows processed


In [ ]:
# TODO
rows_processed = None


### Exercise 12.1 (Checks)


In [ ]:
assert isinstance(rows_processed, int)
assert rows_processed == large_df.shape[0]


## 13) Mini-lab


In [ ]:

def run_pipeline(raw_dir: Path, bronze_dir: Path, silver_dir: Path, output_dir: Path) -> Dict[str, Path]:
    people = pd.read_csv(raw_dir / "people.csv")
    people["signup_date"] = pd.to_datetime(people["signup_date"], errors="coerce")
    people["age"] = pd.to_numeric(people["age"], errors="coerce")
    people["salary_usd"] = pd.to_numeric(people["salary_usd"], errors="coerce").fillna(0.0)
    people["email"] = people["email"].fillna("unknown@example.com")
    people["age"] = people["age"].fillna(people["age"].median())
    people = people.drop_duplicates(subset=["person_id"], keep="last")
    people_contract.validate(people)

    events_ok, events_bad = read_jsonl_with_quarantine(raw_dir / "events.jsonl")
    quarantine_out = bronze_dir / "events_quarantine_capstone.csv"
    events_bad.to_csv(quarantine_out, index=False)

    events_ok["event_ts"] = pd.to_datetime(events_ok["event_ts"], errors="coerce")
    if "event_type" not in events_ok.columns:
        events_ok["event_type"] = "unknown"
    else:
        events_ok["event_type"] = events_ok["event_type"].fillna("unknown")

    if events_ok["event_ts"].isna().any():
        raise ValueError("Some events could not be parsed as datetime. Check raw inputs.")
    if events_ok["event_id"].isna().any() or events_ok["person_id"].isna().any():
        raise ValueError("event_id and person_id must not be null")

    enriched = events_ok.merge(
        people[["person_id","city","gender"]],
        on="person_id",
        how="left",
        validate="many_to_one",
    )
    if enriched.shape[0] != events_ok.shape[0]:
        raise ValueError("Join changed row count unexpectedly")

    tmp = enriched.copy()
    tmp["city"] = tmp["city"].fillna("UNKNOWN")
    report = (
        tmp.groupby(["city","event_type"], as_index=False)
           .agg(event_count=("event_id","count"))
           .sort_values(["city","event_type"])
    )

    people_out = silver_dir / "people_capstone.parquet"
    events_out = silver_dir / "events_capstone.parquet"
    report_out = output_dir / "report_capstone.csv"

    people.to_parquet(people_out, index=False)
    events_ok.to_parquet(events_out, index=False)
    report.to_csv(report_out, index=False)

    return {"people_parquet": people_out, "events_parquet": events_out, "quarantine_csv": quarantine_out, "report_csv": report_out}

capstone_artifacts = run_pipeline(RAW_DIR, BRONZE_DIR, SILVER_DIR, OUTPUT_DIR)
capstone_artifacts


### Exercise 13.1 (TODO): Verify capstone outputs


In [ ]:
# TODO
report_total = None
events_total = None


### Exercise 13.1 (Checks)


In [ ]:
report_df = pd.read_csv(capstone_artifacts["report_csv"])
events_capstone_df = pd.read_parquet(capstone_artifacts["events_parquet"])

assert list(report_df.columns) == ["city","event_type","event_count"]
assert report_total == int(report_df["event_count"].sum())
assert events_total == int(events_capstone_df.shape[0])
assert report_total == events_total


> Content created by [**Carlos Cruz-Maldonado**](https://www.linkedin.com/in/carloscruzmaldonado/).  
> I am available to answer any questions or provide further assistance.   
> Feel free to reach out to me at any time.  